# Run CytoBridge on your data

Start with the packaged dataset closest to your experiment, export its config,
and edit the data fields and analysis settings. The same config is then used
for preprocessing, training, downstream calculations, and standard figures.

## Check the input fields

In [1]:
import pandas as pd

pd.DataFrame(
    [
        ("expression", "AnnData X or the count layer named in the config"),
        ("time", "one obs column with a value for every cell"),
        ("cell type", "one obs column used to label generated cells"),
        ("spatial coordinates", "two obs columns or one obsm matrix"),
    ],
    columns=["input", "location"],
)

,input,location
0,expression,AnnData X or the count layer named in the config
1,time,one obs column with a value for every cell
2,cell type,one obs column used to label generated cells
3,spatial coordinates,two obs columns or one obsm matrix


The raw count layer, time mapping, cell-type column, and spatial coordinate
columns must agree with the workflow config. Do not rename fields after model
training; the aligned H5AD and model directory are one matched pair.

## Export a starting config

In [2]:
from pathlib import Path

BASE_PRESET = "zebrafish"
CONFIG_PATH = Path("configs/my_dataset.json")
RAW_H5AD = Path("inputs/my_dataset_raw.h5ad")
RUN_ROOT = Path("outputs/my_dataset")

print(
    f"cytobridge workflow --config {BASE_PRESET} "
    f"--export-config {CONFIG_PATH}"
)

cytobridge workflow --config zebrafish --export-config configs/my_dataset.json


Run the printed command once. In the exported JSON, change these entries before
starting a fit:

- `dataset.name`, `display_name`, and `annotation_key`;
- `preprocess.time_key`, `annotation_source`, count layer, coordinates, and
  `align.time_mapping`;
- `scientific.classifier_k` and the spatial/expression loss weights;
- the training profile, interaction distance, LR database, and predictor
  settings; and
- downstream observed/intermediate times and species tag.

The dataset notebooks show the settings used for the five paper datasets.

## Inspect the run before starting

In [3]:
print(
    "cytobridge workflow "
    f"--config {CONFIG_PATH} --train --input-h5ad {RAW_H5AD} "
    f"--output-dir {RUN_ROOT} --device cuda --dry-run"
)

cytobridge workflow --config configs/my_dataset.json --train --input-h5ad inputs/my_dataset_raw.h5ad --output-dir outputs/my_dataset --device cuda --dry-run


The dry run prints the data keys, training profile, preprocessing outputs,
model directory, downstream time grid, and enabled analyses. Fix missing or
incorrect fields in the JSON before removing `--dry-run`.

## Preprocess, train, and run downstream analysis

In [4]:
print(
    "cytobridge workflow "
    f"--config {CONFIG_PATH} --train --input-h5ad {RAW_H5AD} "
    f"--output-dir {RUN_ROOT} --device cuda"
)

cytobridge workflow --config configs/my_dataset.json --train --input-h5ad inputs/my_dataset_raw.h5ad --output-dir outputs/my_dataset --device cuda


Training is enabled only by `--train`. The command writes the aligned H5AD,
the six-stage model directory, the downstream result folders, a summary file,
and standard PNG/PDF figures under the run root.

## Continue from an existing model

In [5]:
ALIGNED_H5AD = RUN_ROOT / "preprocess" / "my_dataset_aligned.h5ad"
MODEL_DIR = RUN_ROOT / "training"
NEW_DOWNSTREAM = Path("outputs/my_dataset_downstream_rerun")

print(
    "cytobridge workflow "
    f"--config {CONFIG_PATH} --step downstream "
    f"--aligned-h5ad {ALIGNED_H5AD} --model-dir {MODEL_DIR} "
    f"--output-dir {NEW_DOWNSTREAM} --device cuda"
)

cytobridge workflow --config configs/my_dataset.json --step downstream --aligned-h5ad outputs/my_dataset/preprocess/my_dataset_aligned.h5ad --model-dir outputs/my_dataset/training --output-dir outputs/my_dataset_downstream_rerun --device cuda


Use a new output directory for a second downstream run. Paper-figure commands
consume their documented compact schemas; they are not a shortcut for turning
an arbitrary new downstream directory into a manuscript page. Use
`cytobridge figure explain <name>` to check that boundary before reusing one.

## Expected output locations

In [6]:
pd.DataFrame(
    {
        "output": [
            "aligned data",
            "model directory",
            "downstream summary",
            "standard figures",
        ],
        "path": [
            RUN_ROOT / "preprocess" / "my_dataset_aligned.h5ad",
            RUN_ROOT / "training",
            RUN_ROOT / "downstream" / "summary.json",
            RUN_ROOT / "downstream" / "figures",
        ],
    }
)

,output,path
0,aligned data,outputs/my_dataset/preprocess/my_dataset_align...
1,model directory,outputs/my_dataset/training
2,downstream summary,outputs/my_dataset/downstream/summary.json
3,standard figures,outputs/my_dataset/downstream/figures
